# Characterizing the difference between Eastern Oyster transcriptomes in a New York versus Virginia Hatchery

Hello, this little project is based off data from the publication: "Contrasting Immunomodulatory Effects of Probiotic and Pathogenic Bacteria on Eastern Oyster, Crassostrea Virginica, Larvae", doi: 10.3390/vaccines8040588 . This publication compares the transcriptome of different oysters when treated with various probiotics aimed at making them more resistant to a parasitic disease. 

This project will only use the control data 

First we will begin by downloading the raw reads via the SRA toolkit

In [6]:
system("ls", intern = TRUE)
library("IRdisplay")
library(Rhisat2)
library(rtracklayer)

[1] "SRR10982729_1.fastq"                "SRR10982729_2.fastq"               
[3] "eastern_oyster_transcriptome.ipynb"

Now we will fetch our data via the SRA toolkit. 

In [1]:
SRA_list <- c("SRR14150616",  # New Brunswick 1
              "SRR14150615",  # New Brunswick 2  
              "SRR14150614",  # New Brunswick 3
              "SRR14150613",  # New Brunswick 4
              "SRR14150617",  # Louisiana 1
              "SRR14150618",  # Louisiana 2
              "SRR14150619",  # Louisiana 3
              "SRR14150620")  # Louisiana 4

We will prefetch all of our data from the NCBI database. This will mkae the download go much faster than just fasterq dumping everything by dividing the downloading and processing steps. Note that this will use up more temporary space. 

In [4]:
system("vdb-config --prefetch-to-cwd")


for (SRR in SRA_list) {
    system(paste0("prefetch ", SRR))
    }


Now we will do the fasterq dump to get our actual fastq files

In [5]:
for (SRR in SRA_list) {
    system(paste0("fasterq-dump ", SRR))
    }

In [6]:
# making a dedictaed directory 
system("mkdir fastq_files")
system("mv *.fastq fastq_files/")


Now we will run some fastqc in order to do some quality checks on our data.

In [7]:
system("fastqc fastq_files/*.fastq")

#make a directory for the new fastqc files
system("mkdir qc_reports")
system("mv fastq_files/*fastqc.* qc_reports/")

It takes a bit too long to inspect every fastqc files. Using multqc to generate a summary of all fastqc reports is the best way to visualize the overall quality of our data.

In [10]:
# conducts the summary of all theqc reports and moves the  files so that they are the 
system("multiqc qc_reports/")

In [4]:
# system2("conda", args = c("install", "r-irdisplay"))

In [ ]:
library("IRdisplay")
display_html(file = "multiqc_report.html")


## Trimming the Data

Now we will be using the fastp command in order to trim the reads which contain a large amount of adapter sequences. In order to do iterate through all our files we will be using a bash script which I will post below for your convenience:


#!/bin/bash
mkdir -p trimmed_data
cd fastq_files
for r1 in *_1.fastq; do
  r2="${r1%_1.fastq}_2.fastq"
  fastp -i "$r1" -I "$r2" \
        -o "~/e_oyster/project_2/trimmed_data/trimmed_${r1}" -O "~/e_oyster/project_2/trimmed_data/trimmed_${r2}" \
        --html "${r1%.fastq}.html" 
done

In [13]:
system2("conda", args=c("install -c bioconda fastp"))


In [17]:
# This will execute our bash script
system("bash trim_reads.sh")

In [16]:
system("pwd")

In [18]:
system2("fastqc", args= c("trimmed_data/*.fastq"))

In [23]:
# Moving the fastqc reports to the qc reports folder
system2("mkdir", args= c("-p qc_reports/trimmed_reports"))
system2("mv", args= c("raw_data/trimmed_data/*fastqc.*","qc_reports/trimmed_reports/"))

In [24]:
system2("multiqc", args = c("qc_reports/trimmed_reports/"))

Now we will delete the old fastq files to save on some storage space but we will keep the fastqc reports

In [2]:
# be careful with this command
system("rm -r fastq_files")

[View the trimmed multiqc report HERE](multiqc_report_1.html)


If you view the multiqc report, then you will see that the qc report indicates that all of the adapter sequences have been removed from our files. Now we can proceed with our analysis

## HISAT2 alignement

Now that we have our qc's data, we will need to align our transcript data. Since I am conducting this analysis on modest, local hardware I'll use the HISAT2 aligner. If I were doing this analysis on the cloud or on an HPC then I would use STAR for its multithreading capabilities instead.

In [10]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
system2("conda", args = c("install -c bioconda bioconductor-Rhisat2"))


In [2]:
system2("datasets", args= c("download genome taxon Crassostrea virginica --reference --include genome,gff3,rna"))
system2("unzip", args= c("ncbi_dataset.zip"))


In [1]:
library(Rhisat2)

# Build the index from reference fasta files
hisat2_build(
    references = "ncbi_dataset/data/GCF_053477285.1/GCF_053477285.1_ASM5347728v1_genomic.fna", 
    outdir = "ref_seq", 
    prefix = "oyster_index"
)


In [7]:
library(rtracklayer)

Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomeInfoDb



In [55]:
library(rtracklayer)
library(GenomicFeatures)

gff_file <- "ncbi_dataset/data/GCF_053477285.1/genomic.gff"

# 1. Import the GFF3 as a GRanges object
cat("Importing GFF3 file...\n")
gr <- import(gff_file, format="gff3")

# 2. Fix the non-unique transcript names 
# NCBI assigns a unique tag under the 'ID' attribute (e.g., rna-XM_...) 
# We swap the problematic 'transcript_id' with this unique 'ID' field
if ("transcript_id" %in% colnames(mcols(gr))) {
    cat("Fixing duplicate transcript names...\n")
    # Use the unique ID tag where transcript_id exists
    mcols(gr)$transcript_id <- ifelse(!is.na(mcols(gr)$transcript_id), 
                                      mcols(gr)$ID, 
                                      mcols(gr)$transcript_id)
}

# 3. Create the TxDb object from the modified GRanges
cat("Building TxDb object...\n")
txdb <- makeTxDbFromGRanges(gr)

# Save your TxDb object to a file for faster loading in downstream scripts
saveDb(txdb, file="GCF_053477285.1_fixed.txdb")


Importing GFF3 file...
Fixing duplicate transcript names...
Building TxDb object...


Warning message in .extract_exons_from_GRanges(exon_IDX, gr, mcols0, tx_IDX, feature = "exon", :
“23115 exons couldn't be linked to a transcript so were dropped (showing
  only the first 6):
        seqid  start    end strand                    ID                  Name
1 NC_136047.1 211526 213254      + exon-XR_013560623.1-1 exon-XR_013560623.1-1
2 NC_136047.1 214781 215178      + exon-XR_013560623.1-2 exon-XR_013560623.1-2
3 NC_136047.1 568106 568195      + exon-XR_013558991.1-1 exon-XR_013558991.1-1
4 NC_136047.1 573790 574101      + exon-XR_013558991.1-2 exon-XR_013558991.1-2
5 NC_136047.1 574191 574372      + exon-XR_013558991.1-3 exon-XR_013558991.1-3
6 NC_136047.1 568152 568272      + exon-XR_013558987.1-1 exon-XR_013558987.1-1
              Parent Parent_type
1 rna-XR_013560623.1      lncRNA
2 rna-XR_013560623.1      lncRNA
3 rna-XR_013558991.1      lncRNA
4 rna-XR_013558991.1      lncRNA
5 rna-XR_013558991.1      lncRNA
6 rna-XR_013558987.1      lncRNA”
Warning message in .extr

TxDb object:
# Db type: TxDb
# Supporting package: GenomicFeatures
# Genome: NA
# Nb of transcripts: 57707
# Db created by: GenomicFeatures package from Bioconductor
# Creation time: 2026-10-02 15:57:55 -0400 (Fri, 02 Oct 2026)
# GenomicFeatures version at creation time: 1.54.1
# RSQLite version at creation time: 2.4.3
# DBSCHEMAVERSION: 1.2

In [52]:
names(gr) <- as.character((length(gr)))

In [25]:
txdb <- makeTxDbFromGRanges(gr)

ERROR: Error in .makeTxDb_normarg_genes(genes, transcripts$tx_id, transcripts_tx_name = transcripts$tx_name): 'genes$gene_id' must be a character vector (or factor) with no NAs


In [54]:
head(rownames(elementMetadata(mcols(gr))))

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'head': error in evaluating the argument 'x' in selecting a method for function 'rownames': error in evaluating the argument 'x' in selecting a method for function 'elementMetadata': missing values not allowed in rownames


In [14]:
library(GenomicFeatures)

txdb <- makeTxDbFromGFF("ncbi_dataset/data/GCF_053477285.1/genomic.gff", format="gff3")

Loading required package: AnnotationDbi

Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .extract_exons_from_GRanges(exon_IDX, gr, mcols0, tx_IDX, feature = "exon", :
“23115 exons couldn't be linked to a transcript so were dropped (showing
  only the first 6):
        seqid  start    end strand                    ID                  Name
1 NC_136047.1 211526 213254      + exon-XR_013560623.1-1 exon-XR_013560623.1-1
2 NC_136047.1 214781 215178      + exon-XR_013560623.1-2 exon-XR_013560623.1-2
3 NC_136047.1 568106 568195      + exon-XR_013558991.1-1 exon-XR_013558991.1-1
4 NC_136047.1 573790 574101      + exon-XR_013558991.1-2 exon-XR_013558991.1-2
5 

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'rownames': unable to find an inherited method for function ‘mcols’ for signature ‘"TxDb"’


In [1]:
library(rtracklayer)
library(GenomicFeatures)
library(Rhisat2)

Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomeInfoDb

Loa

In [2]:
# Ensure you have 'gffread' installed on your system
# system(" conda install -c bioconda gffread ")
system("gffread ncbi_dataset/data/GCF_053477285.1/genomic.gff -F -T -o ncbi_dataset/data/GCF_053477285.1/converted_genomic.gtf")

In [30]:
system("pip install lckr-jupyterlab-variableinspector")

In [3]:

gr <- import("ncbi_dataset/data/GCF_053477285.1/converted_genomic.gtf")


In [25]:
system("head ncbi_dataset/data/GCF_053477285.1/converted_genomic.gtf")

In [21]:
na_trans <- is.na(gr$transcript_id) & !is.na(gr$ID)
head(gr$ID[na_trans])

NULL

In [10]:
faulty_ind <- which(is.na(gr$gene_id))
# show why the gene_id must be replaced with the gene name
mcols(gr)[faulty_ind[4],"gene_name"]
mcols(gr)[faulty_ind[4],"transcript_id"]

mcols(gr)[faulty_ind, "gene_id"] <- mcols(gr)[faulty_ind, "gene_name"] 

[1] "LOC144618302"

[1] "gene-LOC144618302"

In [11]:
# 2. Build the clean TxDb object
txdb <- makeTxDbFromGRanges(gr)

In [ ]:
extract_splice_sites(features = txdb, outfile = "splice_sites.txt")


In [13]:
export(gr,  con = "ncbi_dataset/data/GCF_053477285.1/edited_genomic.gtf", format = "gtf")

In [34]:
extract_splice_sites(features = gr, outfile = "splice_sites.txt")

In [10]:
gr

GRanges object with 1350596 ranges and 47 metadata columns:
               seqnames            ranges strand |   source       type
                  <Rle>         <IRanges>  <Rle> | <factor>   <factor>
        [1] NC_136047.1        1-64991568      + |   RefSeq     region
        [2] NC_136047.1       34876-71533      + |   Gnomon     gene  
        [3] NC_136047.1       34876-71533      + |   Gnomon     mRNA  
        [4] NC_136047.1       34876-35035      + |   Gnomon     exon  
        [5] NC_136047.1       46195-46780      + |   Gnomon     exon  
        ...         ...               ...    ... .      ...        ...
  [1350592] NC_136056.1 36889809-36890104      - |   RefSeq cDNA_match
  [1350593] NC_136056.1 36886389-36887349      - |   RefSeq cDNA_match
  [1350594] NC_136056.1 36886240-36886302      - |   RefSeq cDNA_match
  [1350595] NC_136056.1 36886108-36886155      - |   RefSeq cDNA_match
  [1350596] NC_136056.1 36881535-36884853      - |   RefSeq cDNA_match
                s

In [31]:
length(which(is.na(mcols(gr)$gene_id)))

[1] 2412

In [33]:
gr <- gr[!which(is.na(mcols(gr)$gene_id))]

In [18]:
hi <- "hi"
ls()

[1] "hi"

In [36]:
r1_files <- list.files(path = "trimmed_data",  pattern = "_1.*\\.(fastq|fq)$", full.names = TRUE)
r2_files <- list.files(path = "trimmed_data", pattern = "_2.*\\.(fastq|fq)$", full.names = TRUE)

r1_files <- sort(r1_files)
r2_files <- sort(r2_files)

paired_files <- list()
for (i in 1:length(r1_files)){
    paired_files[[i]] <-  list(r1_files[i], r2_files[i])
    }
paired_files


[[1]]
[[1]][[1]]
[1] "trimmed_data/trimmed_SRR14150613_1.fastq"

[[1]][[2]]
[1] "trimmed_data/trimmed_SRR14150613_2.fastq"


[[2]]
[[2]][[1]]
[1] "trimmed_data/trimmed_SRR14150614_1.fastq"

[[2]][[2]]
[1] "trimmed_data/trimmed_SRR14150614_2.fastq"


[[3]]
[[3]][[1]]
[1] "trimmed_data/trimmed_SRR14150615_1.fastq"

[[3]][[2]]
[1] "trimmed_data/trimmed_SRR14150615_2.fastq"


[[4]]
[[4]][[1]]
[1] "trimmed_data/trimmed_SRR14150616_1.fastq"

[[4]][[2]]
[1] "trimmed_data/trimmed_SRR14150616_2.fastq"


[[5]]
[[5]][[1]]
[1] "trimmed_data/trimmed_SRR14150617_1.fastq"

[[5]][[2]]
[1] "trimmed_data/trimmed_SRR14150617_2.fastq"


[[6]]
[[6]][[1]]
[1] "trimmed_data/trimmed_SRR14150618_1.fastq"

[[6]][[2]]
[1] "trimmed_data/trimmed_SRR14150618_2.fastq"


[[7]]
[[7]][[1]]
[1] "trimmed_data/trimmed_SRR14150619_1.fastq"

[[7]][[2]]
[1] "trimmed_data/trimmed_SRR14150619_2.fastq"


[[8]]
[[8]][[1]]
[1] "trimmed_data/trimmed_SRR14150620_1.fastq"

[[8]][[2]]
[1] "trimmed_data/trimmed_SRR14150620_2.fastq"

character(0)

In [14]:
for (path in paired_files) {
    
    hisat2(
        sequences = path, 
        index = file.path("ref_seq", "oyster_index"), 
        type = "paired", 
        outfile = paste0("aligned_", cur_SRR, ".sam"),
        force = TRUE,
        strict = TRUE,
        execute = TRUE,
        `known-splicesite-infile` = "splice_sites.txt",
        p = 4  # number of threads
    )
    }

In [1]:
file.path("ref_seq", "oyster_index")

[1] "ref_seq/oyster_index"

In [4]:
library(Rhisat2)

In [7]:
file.path("ref_seq", "oyster_index")

[1] "ref_seq/oyster_index"

In [8]:
path

[[1]]
[1] "raw_data/trimmed_data/trimmed_SRR10982743_1.fastq"

[[2]]
[1] "raw_data/trimmed_data/trimmed_SRR10982743_2.fastq"

In [27]:
library(Rsamtools)
sam_files <- list.files(path = ".", pattern = "\\.sam$")


for (f in sam_files) {
    dest <- paste0("bam_", tools::file_path_sans_ext(f))
    asBam(f, overwrite = TRUE, destination = dest )
}



[1] "bam_aligned_trimmed_SRR10982729_1"
[1] "bam_aligned_trimmed_SRR10982739_1"
[1] "bam_aligned_trimmed_SRR10982743_1"


In [32]:
system("mkdir sorted_bam_files")
system("mv *.bam* sorted_bam_files/")


In [40]:

bam_files <- list.files(path = "sorted_bam_files", pattern = "\\.bam$", full.names = TRUE)
#print(bam_files)

[1] "sorted_bam_files/bam_aligned_trimmed_SRR10982729_1.bam"
[2] "sorted_bam_files/bam_aligned_trimmed_SRR10982739_1.bam"
[3] "sorted_bam_files/bam_aligned_trimmed_SRR10982743_1.bam"


In [42]:
library(Rsubread)

counts <- featureCounts(
  files = bam_files, 
  annot.ext = "ncbi_dataset/data/GCF_053477285.1/edited_genomic.gtf", 
  isGTFAnnotationFile = TRUE,
  GTF.featureType = "exon",      # Level to count (usually exon)
  GTF.attrType = "transcript_id",      # Attribute to aggregate by (usually gene_id)
  isPairedEnd = TRUE             # Change to FALSE if your data is single-end
)

# Extract the count matrix
#gene_counts <- counts$counts


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.16.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 3 BAM files                                      ||
||                                                                            ||
||                           bam_aligned_trimmed_SRR10982729_1.bam            ||
||                           bam_aligned_trimmed_SRR10982739_1.bam            ||
||                           bam_aligned_trimmed_SRR

In [35]:
system("conda install bioconda::bioconductor-rsubread")


In [43]:
gene_counts <- counts$counts
print(colnames(gene_counts))

[1] "bam_aligned_trimmed_SRR10982729_1.bam"
[2] "bam_aligned_trimmed_SRR10982739_1.bam"
[3] "bam_aligned_trimmed_SRR10982743_1.bam"


In [44]:
system("conda install bioconda::bioconductor-edgeR")

In [45]:
library(edgeR)
# Assume 'counts' is a numeric matrix (genes in rows, samples in columns)
# Assume 'group' is a factor or vector indicating conditions (e.g., c("control", "control", "treated", "treated"))
 groups <- c("virginia", "New_York", "control_virginia")
y <- DGEList(counts = gene_counts, group = groups)


Loading required package: limma


Attaching package: ‘limma’


The following object is masked from ‘package:BiocGenerics’:

    plotMA




In [51]:
low_expression_rows <- filterByExpr(y)
filtered_y <- y[low_expression_rows, , keep.lib.sizes = FALSE]


In [ ]:
calculate the normalization factor by taking into the account the overall size of the library. 
We must normalize how much expression of any one gene is happening by comparing against how much all genees are expressed within the library

In [52]:
filtered_y <- calcNormFactors(filtered_y)
design <- model.matrix(~ groups)


In [ ]:
Estimate the dispersion

In [53]:
filtered_y <- estimateDisp(filtered_y, design)


Warning message in estimateDisp.default(y = y$counts, design = design, group = group, :
“No residual df: setting dispersion to NA”


In [ ]:
# Fit the quasi-likelihood negative binomial generalized log-linear model
fit <- glmQLFit(filtered_y, design)

# Conduct the quasi-likelihood F-test for the second coefficient (e.g., treated vs control)
qlf <- glmQLFTest(fit, coef = 2)

# View top differentially expressed genes
topTags(qlf)

# Extract a full table of results
res <- topTags(qlf, n = 20)$table
